# Homework 5 — solution by (your_name)

Read the homework PDF first: it has the rules. The questions are quoted in this notebook, in blockquotes (with a bar on
their left), so that your answers stand apart. Copy this notebook to
`3dasm_course/Assignments/your_Assignments/HW5/your_Homework5.ipynb` and work there.

**How this notebook works** (as in the lectures and in Homework 4):

1. the cell `%%writefile CLAUDE.md` writes the instructions file that every agent of this homework reads: **you** write it;
2. each prompt is a `PROMPT_...` cell, run with `run_agent`: **you** write the prompts. Every agent works in its own
   task folder inside `HW5/` (e.g. `HW5/ex4_2/`);
3. after each prompt, a cell that we provide loads the agent's `results.json`. The keys it needs are written just above
   each prompt cell: your prompt must make the agent store exactly those. If one is missing, the cell tells you which.

**Your prompts are graded** (see the PDF): write each one in your own words, and make it define the problem uniquely.
A prompt that only points to a question ("solve exercise 4.2"), or pastes it, counts as not answered.

In [ ]:
# Provided by the course: the helper that runs a coding agent from this notebook (see agent_helper.py)
import sys; sys.path.insert(0, "../../..")          # the folder 3dasm_course, where agent_helper.py is
from agent_helper import set_model, run_agent, results, figures, show, save_results, task_folder
import numpy as np, pandas as pd

set_model("sonnet")        # "sonnet" or "haiku": the TA runs your notebook again with this same model
import sklearn; print("numpy", np.__version__, "| scikit-learn", sklearn.__version__)   # some results depend on them

## Your instructions file for the agents

Write here everything that is the same for all prompts: the environment, the packages, where files go, how randomness
must be handled, and what the agent must report. The lectures and Homework 4 have examples.

In [ ]:
%%writefile CLAUDE.md
# YOUR CLAUDE.md HERE

## Exercise 1 (pen and paper)

Consider Least Squares, Ridge and Lasso regression as derived in class, with $M$ basis functions that are **orthonormal on the training points**, $\boldsymbol{\Phi}^T\boldsymbol{\Phi} = \mathbf{I}_M$, and no separate bias term. Each method gives a **point estimate** of the weights: $\hat{\mathbf{w}}_{\text{LS}}$ (the MLE), $\hat{\mathbf{w}}_{\text{ridge}}$ and $\hat{\mathbf{w}}_{\text{lasso}}$ (MAP estimates). In this exercise you compare them analytically.

> **1.1.** Show that $\text{RSS}(\mathbf{w}) = \text{RSS}(\hat{\mathbf{w}}_{\text{LS}}) + ||\mathbf{w} - \hat{\mathbf{w}}_{\text{LS}}||_2^2$, where here $\text{RSS}(\mathbf{w}) = ||\mathbf{y} - \boldsymbol{\Phi}\mathbf{w}||_2^2$. (Hint: write $\mathbf{y} - \boldsymbol{\Phi}\mathbf{w} = (\mathbf{y} - \boldsymbol{\Phi}\hat{\mathbf{w}}_{\text{LS}}) + \boldsymbol{\Phi}(\hat{\mathbf{w}}_{\text{LS}} - \mathbf{w})$, and use the condition that defines the Least Squares estimate in Lecture 9, $\boldsymbol{\Phi}^T\boldsymbol{\Phi}\,\hat{\mathbf{w}}_{\text{LS}} = \boldsymbol{\Phi}^T\mathbf{y}$.) This result says that the RSS of any $\mathbf{w}$ is the smallest possible RSS plus the squared distance between $\mathbf{w}$ and $\hat{\mathbf{w}}_{\text{LS}}$: Ridge and Lasso trade this distance against their penalty, which lets you find and compare their point estimates in closed form.

*YOUR ANSWER HERE*

> **1.2.** Using 1.1, show that the point estimate of Ridge (minimizing $\text{RSS}(\mathbf{w}) + \alpha||\mathbf{w}||_2^2$) is $\hat{\mathbf{w}}_{\text{ridge}} = \hat{\mathbf{w}}_{\text{LS}}/(1+\alpha)$. (Hint: Lecture 9 used $\frac{\partial\, \mathbf{u}^T\mathbf{A}\mathbf{u}}{\partial \mathbf{u}} = (\mathbf{A} + \mathbf{A}^T)\mathbf{u}$; here $\mathbf{A} = \mathbf{I}$.) Can a weight become exactly zero?

*YOUR ANSWER HERE*

> **1.3.** Using 1.1, show that the point estimate of Lasso (minimizing $\text{RSS}(\mathbf{w}) + \lambda||\mathbf{w}||_1$) is, for each weight, $\hat{w}_{\text{lasso},m} = \text{sign}(\hat{w}_{\text{LS},m})\,\max\left(|\hat{w}_{\text{LS},m}| - \lambda/2,\ 0\right)$, called **soft-thresholding**. (Hint: write the objective with indices, $||\mathbf{w} - \hat{\mathbf{w}}_{\text{LS}}||_2^2 = \sum_m (w_m - \hat{w}_{\text{LS},m})^2$ and $||\mathbf{w}||_1 = \sum_m |w_m|$: it becomes a sum of $M$ problems with one weight each. For one weight, consider $w_m > 0$, $w_m < 0$ and $w_m = 0$.) Sketch the point estimate $\hat{w}_m$ as a function of $\hat{w}_{\text{LS},m}$ for Ridge and for Lasso.

*YOUR ANSWER HERE*

> **1.4.** As $\lambda$ grows, which weights does Lasso set to zero first? What happens to the weights that remain non-zero? Relate both to the shape of the Laplace prior compared with the Gaussian prior.

*YOUR ANSWER HERE*

> **1.5.** In Ridge regression with a complete polynomial basis of degree $p$, which quantities are parameters and which are hyperparameters? Why should the hyperparameters be chosen with cross-validation on the training set, and not with the test set?

*YOUR ANSWER HERE*

## Exercise 2 (pen and paper)

Consider Linear Least Squares with $M$ basis functions and $N$ training points at fixed inputs $\mathbf{x}_1, \dots, \mathbf{x}_N$, with the matrix of basis functions $\boldsymbol{\Phi}$ ($N\times M$, with $\boldsymbol{\Phi}^T\boldsymbol{\Phi}$ invertible). Assume that the model can represent the true function, $f(\mathbf{x}) = \boldsymbol{\phi}(\mathbf{x})^T\mathbf{w}$, so that $\mathbf{y} = \boldsymbol{\Phi}\mathbf{w} + \boldsymbol{\epsilon}$, with independent noise $\epsilon_n \sim \mathcal{N}(0, \sigma^2)$.

> **2.1.** Show that the point estimate $\hat{\mathbf{w}} = (\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}\boldsymbol{\Phi}^T\mathbf{y}$ is unbiased, i.e. that $\mathbb{E}[\hat{\mathbf{w}}] = \mathbf{w}$, and that its covariance matrix is $\text{Cov}[\hat{\mathbf{w}}] = \sigma^2(\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}$. (Hint: $\text{Cov}[\hat{\mathbf{w}}] = \mathbb{E}\left[(\hat{\mathbf{w}} - \mathbf{w})(\hat{\mathbf{w}} - \mathbf{w})^T\right]$ and $\mathbb{E}[\boldsymbol{\epsilon}\boldsymbol{\epsilon}^T] = \sigma^2\mathbf{I}_N$.)

*YOUR ANSWER HERE*

> **2.2.** Show that the variance of the prediction $\hat{y}(\mathbf{x}) = \boldsymbol{\phi}(\mathbf{x})^T\hat{\mathbf{w}}$ at an input $\mathbf{x}$ is $\sigma^2\,\boldsymbol{\phi}(\mathbf{x})^T(\boldsymbol{\Phi}^T\boldsymbol{\Phi})^{-1}\boldsymbol{\phi}(\mathbf{x})$. What is the bias of this prediction, $\text{Bias}[\hat{y}(\mathbf{x})] = \mathbb{E}[\hat{y}(\mathbf{x})] - f(\mathbf{x})$?

*YOUR ANSWER HERE*

> **2.3.** The variance of 2.2, averaged over the $N$ training inputs, is $\sigma^2 M/N$ (if you are curious: each term is a number, equal to its own trace, and $\text{tr}(\mathbf{A}\mathbf{B}) = \text{tr}(\mathbf{B}\mathbf{A})$), and the expected training $\text{MSE}$ is $\sigma^2(N-M)/N$ (Lectures 9 and 11). Now consider new observations at the same inputs, $y_n^* = f(\mathbf{x}_n) + \epsilon_n^*$, with new noise $\epsilon_n^*$ independent of the training data. Show that the expected $\text{MSE}$ of the predictions on these new observations is $\sigma^2(1 + M/N)$. (Hint: write $y_n^* - \hat{y}(\mathbf{x}_n) = \epsilon_n^* + \left(f(\mathbf{x}_n) - \hat{y}(\mathbf{x}_n)\right)$; this is the decomposition of the error on unseen data of Lecture 11.) Compare the expected training $\text{MSE}$ with the expected $\text{MSE}$ on new observations: what happens to each when $M$ grows with $N$ fixed, and when $N$ grows with $M$ fixed? Which terms of the decomposition (noise, bias$^2$, variance) appear here, which one is missing, and why?

*YOUR ANSWER HERE*

## Exercise 3 (pen and paper)

Consider Bayesian linear regression with $M$ basis functions, $f(\mathbf{x}) = \mathbf{w}^T\boldsymbol{\phi}(\mathbf{x})$, and the prior $p(\mathbf{w}) = \mathcal{N}(\mathbf{w}|\mathbf{0}, \overset{\scriptscriptstyle <}{\sigma}_w^2\mathbf{I}_M)$. You will need the following property, which follows from the rules of Lecture 3 (the expected value is linear; covariances transform with the matrix): if $\mathbf{w} \sim \mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ and $\mathbf{A}$ is a fixed matrix, then $\mathbf{A}\mathbf{w} \sim \mathcal{N}(\mathbf{A}\boldsymbol{\mu}, \mathbf{A}\boldsymbol{\Sigma}\mathbf{A}^T)$.

> **3.1.** Show that the function values at $N$ inputs, $\mathbf{f} = \boldsymbol{\Phi}\mathbf{w}$, follow $\mathcal{N}(\mathbf{f}|\mathbf{0}, \overset{\scriptscriptstyle <}{\sigma}_w^2\boldsymbol{\Phi}\boldsymbol{\Phi}^T)$. This result shows that, for two inputs $\mathbf{x}$ and $\mathbf{x}'$, the function values $f(\mathbf{x})$ and $f(\mathbf{x}')$ are jointly Gaussian, with zero mean and $\text{Cov}[f(\mathbf{x}), f(\mathbf{x}')] = \overset{\scriptscriptstyle <}{\sigma}_w^2\,\boldsymbol{\phi}(\mathbf{x})^T\boldsymbol{\phi}(\mathbf{x}')$.

*YOUR ANSWER HERE*

> **3.2.** The rank of the $N \times N$ covariance matrix $\overset{\scriptscriptstyle <}{\sigma}_w^2\boldsymbol{\Phi}\boldsymbol{\Phi}^T$ is $\text{rank}(\boldsymbol{\Phi}) \le \min(N, M)$. What does this imply, when $N > M$, about the functions that this prior can generate?

*YOUR ANSWER HERE*

> **3.3.** Write the variance of the PPD at a new input $\mathbf{x}^*$ (Lecture 13), and identify its aleatoric and its epistemic part. What happens to each part when the number of training points grows? (Hint: $\boldsymbol{\Phi}^T\boldsymbol{\Phi} = \sum_n \boldsymbol{\phi}(\mathbf{x}_n)\boldsymbol{\phi}(\mathbf{x}_n)^T$ has one term per training point; for training points spread over the same domain, it grows like $N\mathbf{C}$, where $\mathbf{C}$ is a fixed matrix, the average of $\boldsymbol{\phi}(\mathbf{x})\boldsymbol{\phi}(\mathbf{x})^T$ over the inputs.) And what happens to each part when $\overset{\scriptscriptstyle <}{\sigma}_w \to 0$?

*YOUR ANSWER HERE*

## Exercise 4 (with the agent)

In this exercise you measure the decomposition of the error on unseen data seen in class, by drawing many training sets. Consider $f(x) = x\sin(x)$ in $[0, 10]$ and noise with standard deviation $\sigma = 1$. A **training set** of size $N$ has the inputs $x$ = `np.linspace(0, 10, N)` (always the same inputs) and the outputs $y = f(x) + \epsilon$, with a new noise draw $\epsilon \sim \mathcal{N}(0, 1)$ for every training set. For each $N$, draw 1000 training sets one after the other from a new NumPy default random number generator with seed 0 (the noise of training set $r$ is the $r$-th draw of $N$ values; training set 1 is the **first training set**). Use polynomial models with the inputs rescaled to $[-1, 1]$, and evaluate every model on 201 equally spaced points of $[0, 10]$.

> **4.1.** **Before running anything**, sketch how you expect the bias$^2$, the variance and their sum with the noise to change with the polynomial degree, for $N = 20$ and for $N = 80$.

*YOUR ANSWER HERE*

> **4.2.** For $N = 20$ and $N = 80$, and for degrees 1, 3, 5, 7, 9, 12 and 15 (Linear Least Squares), estimate the bias$^2$ and the variance of the predictions (each averaged over the 201 evaluation points), and the expected error on new observations, noise + bias$^2$ + variance. For degree 7 and $N = 20$, check this sum against the $\text{MSE}$ of the predictions on new noisy observations at the evaluation points (one draw of 201 noise values per training set, from a new generator with seed 1). Plot the three curves against the degree for both $N$.

Your prompt must make the agent store in `results.json`: `bias2`, `variance` and `total` (each a 2×7 list: rows $N$ = 20, 80; columns degree 1, 3, 5, 7, 9, 12, 15), and `check_sum` and `check_mse_new` (the two numbers of the check, for degree 7 and $N$ = 20).

In [ ]:
TASK_4_2 = "ex4_2"          # the task name (the cells below use it)
PROMPT_4_2 = """YOUR PROMPT HERE"""
run_agent(TASK_4_2, PROMPT_4_2)

In [ ]:
# Provided: the results of task ex4_2
r = results(TASK_4_2, ["bias2", "variance", "total", "check_sum", "check_mse_new"])
degrees = [1, 3, 5, 7, 9, 12, 15]
for i, N in enumerate((20, 80)):
    print(f"N = {N}\n", pd.DataFrame({k: r[k][i] for k in ("bias2", "variance", "total")}, index=[f"deg {d}" for d in degrees]).to_string(float_format="%.3f"), "\n")
print("check (degree 7, N = 20): noise + bias2 + variance =", round(r["check_sum"], 3), "| MSE on new observations =", round(r["check_mse_new"], 3))
show(*figures(TASK_4_2), width=800)

*YOUR ANSWER HERE*

> **4.3.** Check the average variance $\sigma^2 M/N$ of 2.3: for degrees 3 and 7 with $N = 20$, and 7 and 12 with $N = 80$, compare the variance averaged over the **training inputs** with $\sigma^2 M/N$.

Your prompt must make the agent store in `results.json`: `measured` and `formula` (each a list of 4 values, for ($N$, degree) = (20, 3), (20, 7), (80, 7), (80, 12)).

In [ ]:
TASK_4_3 = "ex4_3"          # the task name (the cells below use it)
PROMPT_4_3 = """YOUR PROMPT HERE"""
run_agent(TASK_4_3, PROMPT_4_3)

In [ ]:
# Provided: the results of task ex4_3
r = results(TASK_4_3, ["measured", "formula"])
print(pd.DataFrame(r, index=["N=20, deg 3", "N=20, deg 7", "N=80, deg 7", "N=80, deg 12"]).to_string(float_format="%.4f"))

*YOUR ANSWER HERE*

> **4.4.** For degree 12 and $N = 20$, repeat 4.2 with Ridge regression for $\alpha$ = $10^{-6}$, $10^{-4}$, $10^{-3}$, $10^{-2}$, $10^{-1}$, $1$ and $10$, and plot the curves against $\alpha$.

Your prompt must make the agent store in `results.json`: `alpha` (the 7 values), and `bias2`, `variance` and `total` (each a list of 7 values, in the order of `alpha`).

In [ ]:
TASK_4_4 = "ex4_4"          # the task name (the cells below use it)
PROMPT_4_4 = """YOUR PROMPT HERE"""
run_agent(TASK_4_4, PROMPT_4_4)

In [ ]:
# Provided: the results of task ex4_4
r = results(TASK_4_4, ["alpha", "bias2", "variance", "total"])
print(pd.DataFrame({k: r[k] for k in ("bias2", "variance", "total")}, index=[f"alpha {a:g}" for a in r["alpha"]]).to_string(float_format="%.3f"))
show(*figures(TASK_4_4), width=600)

*YOUR ANSWER HERE*

> **4.5.** Now use only the **first** training set of each size, as you would in practice: choose the degree with 5-fold cross-validation (points shuffled, seed 123). Which degree does it choose for $N = 20$ and for $N = 80$, compared with the minimum of 4.2?

Your prompt must make the agent store in `results.json`: `cv_mse` (a 2×7 list: rows $N$ = 20, 80; columns degree 1, 3, 5, 7, 9, 12, 15) and `chosen_degree` (a list of 2 values, for $N$ = 20, 80).

In [ ]:
TASK_4_5 = "ex4_5"          # the task name (the cells below use it)
PROMPT_4_5 = """YOUR PROMPT HERE"""
run_agent(TASK_4_5, PROMPT_4_5)

In [ ]:
# Provided: the results of task ex4_5
r = results(TASK_4_5, ["cv_mse", "chosen_degree"])
print(pd.DataFrame(r["cv_mse"], index=["N=20", "N=80"], columns=[f"deg {d}" for d in (1, 3, 5, 7, 9, 12, 15)]).to_string(float_format="%.2f"))
print("chosen degree:", dict(zip(("N=20", "N=80"), r["chosen_degree"])))

*YOUR ANSWER HERE*

> **4.6.** For degree 12 and $N = 20$, repeat the variance estimate with inputs taken from a single uniform draw of 20 values in $[0, 10]$ (a NumPy default random number generator with seed 123, kept fixed for all training sets) instead of equally spaced; the noise is drawn as before. Plot the variance against $x$ for both choices of inputs. Where is it large, and why?

Your prompt must make the agent store in `results.json`: `variance_equal` and `variance_uniform` (each a list of 3 values: the variance at $x$ = 0, 5 and 10).

In [ ]:
TASK_4_6 = "ex4_6"          # the task name (the cells below use it)
PROMPT_4_6 = """YOUR PROMPT HERE"""
run_agent(TASK_4_6, PROMPT_4_6)

In [ ]:
# Provided: the results of task ex4_6
r = results(TASK_4_6, ["variance_equal", "variance_uniform"])
print(pd.DataFrame(r, index=["x = 0", "x = 5", "x = 10"]).to_string(float_format="%.3g"))
show(*figures(TASK_4_6), width=600)

*YOUR ANSWER HERE*

> **4.7.** **What can you conclude?** Compare with your predictions of 4.1..

*YOUR ANSWER HERE*

## Exercise 5 (with the agent)

In this exercise the noise changes with the input (heteroscedastic noise, Lecture 13), and you know the truth, so you can check every step. Consider $f(x) = 2\sin(x) + 0.5x$ in $[0, 10]$ and noise with standard deviation $\sigma(x) = 0.2\,e^{0.25x}$. A **training set** with seed $s$ has $N = 60$ inputs drawn uniformly in $[0, 10]$ and outputs $y = f(x) + \epsilon$, $\epsilon \sim \mathcal{N}\left(0, \sigma^2(x)\right)$, both from a NumPy default random number generator with seed $s$ (first the 60 inputs, then the 60 noise values). Unless said otherwise, use the training set with seed 0. The **test set** has 5000 points drawn in the same way with seed 1. The model of the mean is Bayesian linear regression with a polynomial of degree 7 in $\tilde{x} = x/5 - 1$ (8 basis functions, including the constant) and the prior $p(\mathbf{w}) = \mathcal{N}(\mathbf{w}|\mathbf{0}, 10^2\mathbf{I})$ on all the weights. The **coverage** of a model is the fraction of test points with $|y - \text{PPD mean}| \le 1.96\ \text{PPD standard deviation}$, computed separately in each third of the domain, $[0, 10/3)$, $[10/3, 20/3)$ and $[20/3, 10]$.

> **5.1.** **Before running anything**, predict the coverage in each third for a model that assumes one noise level $\sigma$ for all points.

*YOUR ANSWER HERE*

> **5.2.** Fit the model with one noise level (homoscedastic Bayesian linear regression), estimating $\sigma$ by alternating: start with $\sigma = 1$ and repeat 50 times: compute the posterior of $\mathbf{w}$ with the current $\sigma$, then update $\hat\sigma^2 = \sum_n (y_n - \hat\mu(x_n))^2/(N - M)$, where $\hat\mu$ is the PPD mean. Report $\hat\sigma$ and the coverage in each third, and plot the training points, $f$, the PPD mean and two bands of $\pm 2$ standard deviations (aleatoric only, and aleatoric + epistemic).

Your prompt must make the agent store in `results.json`: `sigma_hat` (the estimated noise standard deviation) and `coverage` (a list of 3 values, one per third).

In [ ]:
TASK_5_2 = "ex5_2"          # the task name (the cells below use it)
PROMPT_5_2 = """YOUR PROMPT HERE"""
run_agent(TASK_5_2, PROMPT_5_2)

In [ ]:
# Provided: the results of task ex5_2
r = results(TASK_5_2, ["sigma_hat", "coverage"])
print("sigma_hat =", round(r["sigma_hat"], 4), "| coverage by thirds:", np.round(r["coverage"], 3))
show(*figures(TASK_5_2), width=650)

*YOUR ANSWER HERE*

> **5.3.** Now model the noise with $\log\sigma^2(x) = v_0 + v_1\tilde{x}$, and solve the model with the staggered scheme of Lecture 13: start from the result of 5.2 ($v_0 = \log\hat\sigma^2$, $v_1 = 0$) and repeat 10 times: (i) the **noise step**: minimize, with 100 steps of Newton’s method, $\frac{1}{2}\sum_n\left[\mathbf{v}^T\boldsymbol{\psi}(x_n) + \tilde{r}_n^2\,e^{-\mathbf{v}^T\boldsymbol{\psi}(x_n)}\right]$, where $\boldsymbol{\psi}(x) = [1, \tilde{x}]$ and $\tilde{r}_n^2 = \left(y_n - \hat\mu(x_n)\right)^2 + \boldsymbol{\phi}(x_n)^T\overset{\scriptscriptstyle >}{\boldsymbol{\Sigma}}_w\boldsymbol{\phi}(x_n)$ is the squared residual averaged over the posterior of the weights; (ii) the **mean step**: the posterior of $\mathbf{w}$ with the noise variances $\sigma^2(x_n) = e^{\mathbf{v}^T\boldsymbol{\psi}(x_n)}$. Report the estimated $\hat\sigma(x)$ at $x = 0$, 5 and 10 after each iteration, the largest change of the PPD mean (on 201 equally spaced points of $[0, 10]$) from one iteration to the next, and the final coverage in each third. Plot $\hat\sigma(x)$ after each iteration together with the true $\sigma(x)$, and the final PPD as in 5.2..

Your prompt must make the agent store in `results.json`: `sigma_at` (a list of 10 lists of 3 values: $\hat\sigma$ at $x$ = 0, 5 and 10 after each iteration), `mean_change` (a list of 10 values: the largest change of the PPD mean on the 201 points, from the previous iteration; the first from the one-noise-level fit) and `coverage` (a list of 3 values, after the last iteration).

In [ ]:
TASK_5_3 = "ex5_3"          # the task name (the cells below use it)
PROMPT_5_3 = """YOUR PROMPT HERE"""
run_agent(TASK_5_3, PROMPT_5_3)

In [ ]:
# Provided: the results of task ex5_3
r = results(TASK_5_3, ["sigma_at", "mean_change", "coverage"])
print(pd.DataFrame(r["sigma_at"], columns=["x = 0", "x = 5", "x = 10"], index=[f"iteration {k}" for k in range(1, 11)]).assign(
      **{"max change of the mean": r["mean_change"]}).to_string(float_format="%.4f"))
print("true sigma at x = 0, 5, 10:", np.round(0.2 * np.exp(0.25 * np.array([0, 5, 10])), 4))
print("final coverage by thirds:", np.round(r["coverage"], 3))
show(*figures(TASK_5_3), width=650)

*YOUR ANSWER HERE*

> **5.4.** Repeat 5.2 and 5.3 for the 200 training sets with seeds 0 to 199 (the same test set), and report the coverage in each third averaged over the 200 training sets for: one noise level (5.2); the staggered scheme with the plain squared residual $\left(y_n - \hat\mu(x_n)\right)^2$ in the noise step; and the staggered scheme with $\tilde{r}_n^2$ (5.3). Why is the coverage lower with the plain squared residual? (Hint: Lecture 11, the training $\text{MSE}$.)

Your prompt must make the agent store in `results.json`: `coverage_one_sigma`, `coverage_plain` and `coverage_averaged` (each a list of 3 values: the coverage in each third, averaged over the 200 training sets).

In [ ]:
TASK_5_4 = "ex5_4"          # the task name (the cells below use it)
PROMPT_5_4 = """YOUR PROMPT HERE"""
run_agent(TASK_5_4, PROMPT_5_4)

In [ ]:
# Provided: the results of task ex5_4
r = results(TASK_5_4, ["coverage_one_sigma", "coverage_plain", "coverage_averaged"])
print(pd.DataFrame({"one sigma": r["coverage_one_sigma"], "staggered, plain residual": r["coverage_plain"],
                    "staggered, residual averaged over the posterior": r["coverage_averaged"]},
                   index=["[0, 10/3)", "[10/3, 20/3)", "[20/3, 10]"]).T.to_string(float_format="%.3f"))

*YOUR ANSWER HERE*

> **5.5.** **What can you conclude?** Compare with your predictions of 5.1.. Which part of the uncertainty does each step of the staggered scheme estimate? How much does the mean change from one iteration to the next, and how quickly does the scheme converge?

*YOUR ANSWER HERE*

## Exercise 6 (with the agent)

In practice nobody tells you the truth. The file `Assignments/data/HW5_measurements.csv` has 1000 measurements $(x, y)$, with $x \in [0, 10]$, of a quantity whose mean and noise we do not reveal. Split them into training and test sets with `train_test_split` (`test_size=0.25`, `random_state=123`), and use the test set only to evaluate. Here the coverage is computed in each of five bins of the test set, $[0, 2)$, $[2, 4)$, $[4, 6)$, $[6, 8)$ and $[8, 10]$.

> **6.1.** Fit, on the training set, the two models of Exercise 5 with exactly the same choices (a polynomial of degree 7 in $\tilde{x} = x/5 - 1$ for the mean, the prior $\mathcal{N}(\mathbf{0}, 10^2\mathbf{I})$, $\boldsymbol{\psi}(x) = [1, \tilde{x}]$ for the log-variance, and 10 iterations of the staggered scheme with $\tilde{r}_n^2$). Report, for both, the coverage in each bin and the test $\text{MSE}$ of the PPD mean.

Your prompt must make the agent store in `results.json`: `coverage_one_sigma` and `coverage_staggered` (each a list of 5 values, one per bin), and `test_mse_one_sigma` and `test_mse_staggered` (the test MSE of each PPD mean).

In [ ]:
TASK_6_1 = "ex6_1"          # the task name (the cells below use it)
PROMPT_6_1 = """YOUR PROMPT HERE"""
run_agent(TASK_6_1, PROMPT_6_1)

In [ ]:
# Provided: the results of task ex6_1
r = results(TASK_6_1, ["coverage_one_sigma", "coverage_staggered", "test_mse_one_sigma", "test_mse_staggered"])
print(pd.DataFrame({"one sigma": r["coverage_one_sigma"], "staggered, psi = [1, x]": r["coverage_staggered"]},
                   index=["[0, 2)", "[2, 4)", "[4, 6)", "[6, 8)", "[8, 10]"]).T.to_string(float_format="%.2f"))
print("test MSE: one sigma", round(r["test_mse_one_sigma"], 3), "| staggered", round(r["test_mse_staggered"], 3))
show(*figures(TASK_6_1), width=650)

*YOUR ANSWER HERE*

> **6.2.** **Diagnose** the models of 6.1 using only the training set: plot the residuals of the PPD mean of the staggered model, and the squared residuals $\tilde{r}_n^2$ (e.g. on a logarithmic scale), against $x$, together with the estimated $\hat\sigma^2(x)$. What is wrong with the model of the mean, and what is wrong with the model of the noise? Justify with your figures.

Make your diagnosis figures here (with a prompt, or with your own code), and answer below.

In [ ]:
# YOUR DIAGNOSIS HERE (a prompt to the agent, or your own code)

*YOUR ANSWER HERE*

> **6.3.** **Improve** the model: choose the polynomial degree of the mean with 5-fold cross-validation of Linear Least Squares on the training set (`KFold`, points shuffled, `random_state=123`), and choose a basis $\boldsymbol{\psi}$ for the log-variance that you justify with your diagnosis. Report the degree, the coverage in each bin and the test $\text{MSE}$, and plot the final PPD. Is the result perfect? If not, what limits it?

Your prompt must make the agent store in `results.json`: `chosen_degree`, `coverage` (a list of 5 values, one per bin) and `test_mse` (of your final model). This exercise is open: your choices are yours, and they are graded by how well you justify them, not by comparison with the solution.

In [ ]:
TASK_6_3 = "ex6_3"          # the task name (the cells below use it)
PROMPT_6_3 = """YOUR PROMPT HERE"""
run_agent(TASK_6_3, PROMPT_6_3)

In [ ]:
# Provided: the results of task ex6_3
r = results(TASK_6_3, ["chosen_degree", "coverage", "test_mse"])
print("chosen degree:", r["chosen_degree"], "| test MSE:", round(r["test_mse"], 3))
print("coverage by bin:", dict(zip(["[0, 2)", "[2, 4)", "[4, 6)", "[6, 8)", "[8, 10]"], np.round(r["coverage"], 2))))
show(*figures(TASK_6_3), width=650)

*YOUR ANSWER HERE*

> **6.4.** **What can you conclude?** How would you know that a model of the noise is wrong if nobody gave you the truth?

*YOUR ANSWER HERE*

## Exercise 7 (optional, with the agent)

This exercise is optional: do it if you finished the others. It helps us gauge the workload of the homework.

Consider 30 points on the line $y = 1 + 0.5x$: $x$ = `np.linspace(0, 10, 30)` and $y = 1 + 0.5x + \epsilon$, with $\epsilon$ = `np.random.default_rng(0).normal(0, 0.5, 30)`. Then add 10 to the outputs of the points with indices 25, 26 and 27 (three outliers).

> **7.1.** **Before running anything**, predict which of the models below will recover the line $y = 1 + 0.5x$ best, and why.

*YOUR ANSWER HERE*

> **7.2.** Fit a straight line with scikit-learn’s `LinearRegression`, `Ridge(alpha=10)`, `Lasso(alpha=0.5)`, `HuberRegressor()` and `QuantileRegressor(quantile=0.5, alpha=0, solver="highs")`. Report the slope, the intercept and the $\text{MSE}$ between each fitted line and the true line on 201 equally spaced points of $[0, 10]$, and plot the data with the five lines.

Your prompt must make the agent store in `results.json`: `slope`, `intercept` and `mse_true_line` (each a list of 5 values, in the order LinearRegression, Ridge, Lasso, HuberRegressor, QuantileRegressor).

In [ ]:
TASK_7_2 = "ex7_2"          # the task name (the cells below use it)
PROMPT_7_2 = """YOUR PROMPT HERE"""
run_agent(TASK_7_2, PROMPT_7_2)

In [ ]:
# Provided: the results of task ex7_2
r = results(TASK_7_2, ["slope", "intercept", "mse_true_line"])
print(pd.DataFrame(r, index=["LinearRegression", "Ridge(10)", "Lasso(0.5)", "HuberRegressor", "QuantileRegressor(0.5)"]).to_string(float_format="%.3f"))
show(*figures(TASK_7_2), width=600)

*YOUR ANSWER HERE*

> **7.3.** **What can you conclude?** Compare with your predictions of 7.1.. Why do Ridge and Lasso not help? Which observation distribution (likelihood) corresponds to `QuantileRegressor` with `quantile=0.5`, and how does `HuberRegressor` treat the points far from the line?

*YOUR ANSWER HERE*